In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
import os
from glob import glob
from tqdm import tqdm
import random
from torch.utils.data import DataLoader 
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from collections import defaultdict 
import sys # Added for more robust printing during sweep

# --- GLOBAL CONSTANTS & HELPERS ---

# Function to set a fixed seed for all random operations for a specific run
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Constants defining the fixed filter bank span
F_MIN = 500.0
F_MAX = 20000.0
# N_TRNS_REF, Q_CALCULATED, and center_freqs_full are now calculated within the sweep loop.


# ------------------------------
# Trainable Resonant Neuron
# ------------------------------
class TrainableResonantNeuron(nn.Module):
    # initial_Q is now passed from the ResonantNet init
    def __init__(self, initial_f0, initial_Q, duration=0.2, sample_rate=40000):
        super().__init__()
        # Trainable parameters
        self.omega0 = nn.Parameter(torch.tensor(2.0 * np.pi * initial_f0, dtype=torch.float32))
        self.log_Q = nn.Parameter(torch.log(torch.tensor(initial_Q, dtype=torch.float32)))
        
        # Fixed buffers
        self.register_buffer("t", torch.arange(0, duration, 1/sample_rate, dtype=torch.float32))
        
    @property
    def Q(self):
        return torch.exp(self.log_Q)

    def forward(self, freqs, amps, phases):
        t_expanded = self.t.unsqueeze(0).unsqueeze(0).expand(freqs.size(0), -1, -1)
        w_input = (2 * np.pi * freqs).unsqueeze(-1)
        amps_expanded = amps.unsqueeze(-1)
        phases_expanded = phases.unsqueeze(-1)
        
        w0 = self.omega0
        Q = self.Q
        
        w_ratio = w_input.squeeze(-1) / w0
        H_mag_numerator = (w_ratio / Q)
        denominator_term = (w_ratio / Q)**2
        denominator = torch.sqrt((1 - w_ratio**2)**2 + denominator_term + 1e-12)
        H_mag = H_mag_numerator / denominator
        H_mag_expanded = H_mag.unsqueeze(-1)
        
        signal_components = (
            amps_expanded * H_mag_expanded * torch.sin(w_input * t_expanded + phases_expanded)
        )
        
        output_signal = signal_components.sum(dim=1)
        dc_output = torch.abs(output_signal).mean(dim=1)
        return dc_output

# ------------------------------
# ResonantNet
# ------------------------------
class ResonantNet(nn.Module):
    # num_trns and initial_Q are now passed to correctly build the network
    def __init__(self, num_trns, initial_Q, hidden_size=16, clip_value=2.0):
        super().__init__()
        intermediate_points = num_trns + 2
        all_base_freqs = np.logspace(np.log10(F_MIN), np.log10(F_MAX), intermediate_points)
        base_freqs = all_base_freqs[1:-1]
        
        # Instantiate TRNs with dynamically calculated base frequencies and initial Q
        self.trns = nn.ModuleList([
            TrainableResonantNeuron(initial_f0=float(f0), initial_Q=initial_Q) for f0 in base_freqs
        ])
        
        CLIP_ACTIVATION = nn.Hardtanh(min_val=0.0, max_val=clip_value)
        
        # MLP structure depends on the number of TRNs
        self.net = nn.Sequential(
            nn.Linear(num_trns, hidden_size), CLIP_ACTIVATION,
            nn.Linear(hidden_size, hidden_size), CLIP_ACTIVATION,
            nn.Linear(hidden_size, 1)
        )
        
    def forward(self, freqs_batch, amps_batch, phases_batch):
        trn_features = [trn(freqs_batch, amps_batch, phases_batch) for trn in self.trns]
        x = torch.stack(trn_features).transpose(0, 1) 
        return self.net(x).squeeze(1)

# ------------------------------
# Dataset (Unchanged)
# ------------------------------
class SignalDataset(torch.utils.data.Dataset):
    def __init__(self, root_folder="dataset_params", files_subset=None):
        self.files, self.labels = [], []
        
        pos_files = sorted(glob(os.path.join(root_folder, "positive", "*.json")))
        neg_files = sorted(glob(os.path.join(root_folder, "negative", "*.json")))
        
        if files_subset is not None:
            pos_files = [f for f in pos_files if f in files_subset]
            neg_files = [f for f in neg_files if f in files_subset]
            
        self.files += pos_files
        self.labels += [1] * len(pos_files)
        self.files += neg_files
        self.labels += [0] * len(neg_files)
        
        combined = sorted(zip(self.files, self.labels), key=lambda x: x[0])
        self.files = [f for f, l in combined]
        self.labels = [l for f, l in combined]

    def __len__(self):
        return len(self.files)
        
    def __getitem__(self, idx):
        with open(self.files[idx], 'r') as f: data = json.load(f)
        freqs = torch.tensor([d['freq'] for d in data], dtype=torch.float32)
        amps = torch.tensor([d['amp'] for d in data], dtype=torch.float32)
        phases = torch.tensor([d['phase'] for d in data], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return freqs, amps, phases, label

# ------------------------------
# Core Experiment Function
# ------------------------------
def run_experiment(run_id, num_epochs, batch_size, current_N, current_Q):
    """
    Performs one independent training and evaluation run for a specific N and Q.
    """
    # Set seed for this specific run
    set_seed(run_id) 
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    root_folder = "dataset_params"

    # 1. Get POSITIVE and NEGATIVE files and shuffle them separately
    all_pos_files = sorted(glob(os.path.join(root_folder, "positive", "*.json")))
    all_neg_files = sorted(glob(os.path.join(root_folder, "negative", "*.json")))

    random.shuffle(all_pos_files) 
    random.shuffle(all_neg_files) 

    # 2. Determine the size of the splits
    num_pos = len(all_pos_files)
    num_neg = len(all_neg_files)

    train_pos_split = num_pos // 2
    train_neg_split = num_neg // 2

    # 3. Create BALANCED splits
    train_files = all_pos_files[:train_pos_split] + all_neg_files[:train_neg_split]
    random.shuffle(train_files)
    eval_files = all_pos_files[train_pos_split:] + all_neg_files[train_neg_split:]
    random.shuffle(eval_files)

    # 4. Create Datasets and DataLoaders
    dataset_train = SignalDataset(root_folder=root_folder, files_subset=train_files)
    dataset_eval = SignalDataset(root_folder=root_folder, files_subset=eval_files)

    dataloader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
    dataloader_eval = DataLoader(dataset_eval, batch_size=batch_size, shuffle=False)
    
    # 5. Model, Optimizer, Loss - Pass current N and Q
    model = ResonantNet(num_trns=current_N, initial_Q=current_Q, clip_value=2.0).to(device) 
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2) 
    criterion = nn.BCEWithLogitsLoss() 

    print(f"--- Starting Run {run_id+1} (N={current_N}, Initial Q={current_Q:.4f})---", file=sys.stderr)

    # 6. Training Loop
    for epoch in range(num_epochs):
        model.train()
        
        for freqs, amps, phases, label in tqdm(dataloader_train, desc=f"Run {run_id+1}/N{current_N} - Epoch {epoch+1}/{num_epochs} (Train)"):
            freqs, amps, phases, label = freqs.to(device), amps.to(device), phases.to(device), label.to(device)
            
            optimizer.zero_grad()
            output = model(freqs, amps, phases)
            loss = criterion(output, label) 
            
            loss.backward()
            optimizer.step()

    # 7. Final Evaluation
    model.eval()
    all_labels = []
    all_probs = []
    all_preds = []
    eval_loss = 0.0

    with torch.no_grad():
        for freqs, amps, phases, label in dataloader_eval: 
            freqs, amps, phases, label = freqs.to(device), amps.to(device), phases.to(device), label.to(device)
            output = model(freqs, amps, phases)
            
            loss = criterion(output, label) 
            eval_loss += loss.item() * freqs.size(0)
            
            probabilities = torch.sigmoid(output)
            predicted = (probabilities >= 0.5).float()
            
            all_labels.append(label.cpu())
            all_probs.append(probabilities.cpu())
            all_preds.append(predicted.cpu())
            
    final_labels = torch.cat(all_labels).numpy()
    final_probs = torch.cat(all_probs).numpy()
    final_preds = torch.cat(all_preds).numpy()

    total_eval = len(final_labels)
    correct = (final_preds == final_labels).sum()

    # Handle potential metric errors
    try:
        final_auc = roc_auc_score(final_labels, final_probs)
        final_precision = precision_score(final_labels, final_preds, zero_division=0)
        final_recall = recall_score(final_labels, final_preds, zero_division=0)
        final_f1 = f1_score(final_labels, final_preds, zero_division=0)
    except ValueError:
        final_auc, final_precision, final_recall, final_f1 = np.nan, np.nan, np.nan, np.nan


    metrics = {
        'accuracy': correct / total_eval * 100,
        'loss': eval_loss / total_eval,
        'auc': final_auc,
        'precision': final_precision,
        'recall': final_recall,
        'f1': final_f1
    }

    print(f"--- Run {run_id+1} Final Metrics (N={current_N}): Accuracy={metrics['accuracy']:.2f}%, F1={metrics['f1']:.4f} ---", file=sys.stderr)
    return metrics


# ------------------------------
# Main Execution Block for SWEEP AVERAGING
# ------------------------------
BATCH_SIZE = 128
NUM_RUNS = 5
N_SWEEP_RANGE = range(1, 13) # N=1 to N=12 inclusive
ALL_SWEEP_RESULTS = [] # To store the final table data

print("="*80)
print(f"| **STARTING TRN ARCHITECTURE SWEEP: N={min(N_SWEEP_RANGE)} to N={max(N_SWEEP_RANGE)}**")
print("="*80)

for N in N_SWEEP_RANGE:
    # 1. Calculate N-dependent constants (Q and f0s)
    
    # Calculate initial Q according to the formula (Equation 42) for current N:
    # Q = 1 / ((F_MAX/F_MIN)^(1/(2*N)) - (F_MAX/F_MIN)^(-1/(2*N)))
    RATIO = F_MAX / F_MIN
    EXPONENT = 1.0 / (2.0 * N)
    if N == 0: 
        Q_CALCULATED = 1.0 
    else:
        Q_CALCULATED = 1.0 / (RATIO**EXPONENT - RATIO**(-EXPONENT))
    
    intermediate_points = N + 2
    all_base_freqs = np.logspace(np.log10(F_MIN), np.log10(F_MAX), intermediate_points)
    center_freqs = all_base_freqs[1:-1]
    
    # 2. Print Initial Parameters for the current N
    print("\n" + "#"*70)
    print(f"# **Trainable Resonant Neuron (TRN) Parameters for N={N}**")
    print("#" * 70)
    print(f"# Initial Constant Quality Factor (Q) Guess: {Q_CALCULATED:.4f}")
    print(f"# Number of Filters (N): {N}")
    print("#" * 70)
    print("# Initial Center Frequencies (f0, in Hz):")
    print(f"# " + ", ".join([f"{f:.2f}" for f in center_freqs]))
    print("#"*70)
    
    # 3. Execute the 5-run average logic
    all_metrics = defaultdict(list)

    for i in range(NUM_RUNS):
        # 'i' serves as the unique seed for each independent run
        metrics = run_experiment(i, num_epochs=15, batch_size=BATCH_SIZE, current_N=N, current_Q=Q_CALCULATED)
        
        for key, value in metrics.items():
            all_metrics[key].append(value)

    # 4. Calculate and display final statistics for this N
    print("\n" + "="*70)
    print(f"| **AVERAGE EVALUATION METRICS OVER {NUM_RUNS} RUNS (TRN - N={N})**")
    print("-" * 70)

    current_n_results = {'N': N} 

    for key, values in all_metrics.items():
        mean = np.mean(values)
        std = np.std(values)
        
        current_n_results[f'{key}_mean'] = mean
        current_n_results[f'{key}_std'] = std
        
        # Custom print formatting for metrics
        if key == 'accuracy':
            print(f"| Average Accuracy: {mean:.2f}% ± {std:.2f}%")
        elif key == 'loss':
            print(f"| Average Loss: {mean:.4f} ± {std:.4f}")
        else:
            print(f"| Average {key.upper()}: {mean:.4f} ± {std:.4f}")

    print("="*70)
    ALL_SWEEP_RESULTS.append(current_n_results)


# ------------------------------
# FINAL SWEEP RESULTS TABLE
# ------------------------------
print("\n\n" + "#" * 90)
print("# **COMPREHENSIVE SWEEP SUMMARY: TRN Performance (N=1 to N=12)**")
print("#" * 90)

# Print a final summary table in Markdown format
header = "| N | Initial Q | Accuracy (mean ± std) | ROC AUC (mean ± std) | F1 Score (mean ± std) |"
separator = "|---|---|:---:|:---:|:---:|"
print(header)
print(separator)

for result in ALL_SWEEP_RESULTS:
    n = result['N']
    
    # Recalculate Q for the table
    if n == 0:
        q_value = 1.0
    else:
        EXPONENT = 1.0 / (2.0 * n)
        q_value = 1.0 / ((F_MAX/F_MIN)**EXPONENT - (F_MAX/F_MIN)**(-EXPONENT))
        
    acc = f"{result['accuracy_mean']:.2f} ± {result['accuracy_std']:.2f}"
    auc = f"{result['auc_mean']:.4f} ± {result['auc_std']:.4f}"
    f1 = f"{result['f1_mean']:.4f} ± {result['f1_std']:.4f}"
    
    print(f"| {n} | {q_value:.4f} | {acc} | {auc} | {f1} |")
    
print("#" * 90)


| **STARTING TRN ARCHITECTURE SWEEP: N=1 to N=12**

######################################################################
# **Trainable Resonant Neuron (TRN) Parameters for N=1**
######################################################################
# Initial Constant Quality Factor (Q) Guess: 0.1622
# Number of Filters (N): 1
######################################################################
# Initial Center Frequencies (f0, in Hz):
# 3162.28
######################################################################


--- Starting Run 1 (N=1, Initial Q=0.1622)---
Run 1/N1 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:05<00:00, 31.39it/s]
--- Run 1 Final Metrics (N=1): Accuracy=50.00%, F1=0.6667 ---
--- Starting Run 2 (N=1, Initial Q=0.1622)---
Run 2/N1 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:05<00:00, 30.59it/s]
--- Run 2 Final Metrics (N=1): Accuracy=50.00%, F1=0.0000 ---
--- Starting Run 3 (N=1, Initial Q=0.1622)---
Run 3/N1 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:04<00:00, 31.52it/s]
--- Run 3 Final Metrics (N=1): Accuracy=50.00%, F1=0.6667 ---
--- Starting Run 4 (N=1, Initial Q=0.1622)---
Run 4/N1 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:04<00:00, 32.03it/s]
--- Run 4 Final Metrics (N=1): Accuracy=50.00%, F1=0.6667 ---
--- Starting Run 5 (N=1, Initial Q=0.1622)---
Run 5/N1 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:04<00:00, 32.67it/s]
--- Run 5 Final Metrics (N=1): Accuracy=50.00%, F1=0.6667 ---



| **AVERAGE EVALUATION METRICS OVER 5 RUNS (TRN - N=1)**
----------------------------------------------------------------------
| Average Accuracy: 50.00% ± 0.00%
| Average Loss: 0.6932 ± 0.0001
| Average AUC: 0.4995 ± 0.0010
| Average PRECISION: 0.4000 ± 0.2000
| Average RECALL: 0.8000 ± 0.4000
| Average F1: 0.5333 ± 0.2667

######################################################################
# **Trainable Resonant Neuron (TRN) Parameters for N=2**
######################################################################
# Initial Constant Quality Factor (Q) Guess: 0.4723
# Number of Filters (N): 2
######################################################################
# Initial Center Frequencies (f0, in Hz):
# 1709.98, 5848.04
######################################################################


--- Starting Run 1 (N=2, Initial Q=0.4723)---
Run 1/N2 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:04<00:00, 31.60it/s]
--- Run 1 Final Metrics (N=2): Accuracy=50.00%, F1=0.0000 ---
--- Starting Run 2 (N=2, Initial Q=0.4723)---
Run 2/N2 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:05<00:00, 30.90it/s]
--- Run 2 Final Metrics (N=2): Accuracy=50.00%, F1=0.6667 ---
--- Starting Run 3 (N=2, Initial Q=0.4723)---
Run 3/N2 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:05<00:00, 30.51it/s]
--- Run 3 Final Metrics (N=2): Accuracy=50.00%, F1=0.0000 ---
--- Starting Run 4 (N=2, Initial Q=0.4723)---
Run 4/N2 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:05<00:00, 31.09it/s]
--- Run 4 Final Metrics (N=2): Accuracy=50.00%, F1=0.0000 ---
--- Starting Run 5 (N=2, Initial Q=0.4723)---
Run 5/N2 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:05<00:00, 30.62it/s]
--- Run 5 Final Metrics (N=2): Accuracy=50.00%, F1=0.6667 ---



| **AVERAGE EVALUATION METRICS OVER 5 RUNS (TRN - N=2)**
----------------------------------------------------------------------
| Average Accuracy: 50.00% ± 0.00%
| Average Loss: 0.6932 ± 0.0000
| Average AUC: 0.5110 ± 0.0221
| Average PRECISION: 0.2000 ± 0.2449
| Average RECALL: 0.4000 ± 0.4899
| Average F1: 0.2667 ± 0.3266

######################################################################
# **Trainable Resonant Neuron (TRN) Parameters for N=3**
######################################################################
# Initial Constant Quality Factor (Q) Guess: 0.7642
# Number of Filters (N): 3
######################################################################
# Initial Center Frequencies (f0, in Hz):
# 1257.43, 3162.28, 7952.71
######################################################################


...

--- Starting Run 1 (N=11, Initial Q=2.9680)---
Run 1/N11 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:12<00:00, 12.79it/s]
--- Run 1 Final Metrics (N=11): Accuracy=84.06%, F1=0.8550 ---
--- Starting Run 2 (N=11, Initial Q=2.9680)---
Run 2/N11 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:11<00:00, 13.34it/s]
--- Run 2 Final Metrics (N=11): Accuracy=85.87%, F1=0.8720 ---
--- Starting Run 3 (N=11, Initial Q=2.9680)---
Run 3/N11 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:11<00:00, 13.80it/s]
--- Run 3 Final Metrics (N=11): Accuracy=84.21%, F1=0.8603 ---
--- Starting Run 4 (N=11, Initial Q=2.9680)---
Run 4/N11 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:11<00:00, 13.76it/s]
--- Run 4 Final Metrics (N=11): Accuracy=84.17%, F1=0.8586 ---
--- Starting Run 5 (N=11, Initial Q=2.9680)---
Run 5/N11 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:11<00:00, 13.63it/s]
--- Run 5 Final Metrics (N=11): Accuracy=85.14%, F1=0.8658 ---



| **AVERAGE EVALUATION METRICS OVER 5 RUNS (TRN - N=11)**
----------------------------------------------------------------------
| Average Accuracy: 84.69% ± 0.71%
| Average Loss: 0.3524 ± 0.0113
| Average AUC: 0.9000 ± 0.0056
| Average PRECISION: 0.7834 ± 0.0093
| Average RECALL: 0.9591 ± 0.0106
| Average F1: 0.8623 ± 0.0059

######################################################################
# **Trainable Resonant Neuron (TRN) Parameters for N=12**
######################################################################
# Initial Constant Quality Factor (Q) Guess: 3.2402
# Number of Filters (N): 12
######################################################################
# Initial Center Frequencies (f0, in Hz):
# 664.06, 881.94, 1171.32, 1555.65, 2066.08, 2743.99, 3644.33, 4840.09, 6428.19, 8537.36, 11338.59, 15058.95
######################################################################


--- Starting Run 1 (N=12, Initial Q=3.2402)---
Run 1/N12 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:13<00:00, 12.07it/s]
--- Run 1 Final Metrics (N=12): Accuracy=83.02%, F1=0.8463 ---
--- Starting Run 2 (N=12, Initial Q=3.2402)---
Run 2/N12 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:12<00:00, 12.54it/s]
--- Run 2 Final Metrics (N=12): Accuracy=79.82%, F1=0.7886 ---
--- Starting Run 3 (N=12, Initial Q=3.2402)---
Run 3/N12 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:12<00:00, 12.57it/s]
--- Run 3 Final Metrics (N=12): Accuracy=82.25%, F1=0.8423 ---
--- Starting Run 4 (N=12, Initial Q=3.2402)---
Run 4/N12 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:12<00:00, 12.57it/s]
--- Run 4 Final Metrics (N=12): Accuracy=82.50%, F1=0.8385 ---
--- Starting Run 5 (N=12, Initial Q=3.2402)---
Run 5/N12 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:12<00:00, 12.90it/s]



| **AVERAGE EVALUATION METRICS OVER 5 RUNS (TRN - N=12)**
----------------------------------------------------------------------
| Average Accuracy: 81.97% ± 1.11%
| Average Loss: 0.3952 ± 0.0157
| Average AUC: 0.8838 ± 0.0062
| Average PRECISION: 0.7780 ± 0.0267
| Average RECALL: 0.9012 ± 0.0762
| Average F1: 0.8320 ± 0.0219


##########################################################################################
# **COMPREHENSIVE SWEEP SUMMARY: TRN Performance (N=1 to N=12)**
##########################################################################################
| N | Initial Q | Accuracy (mean ± std) | ROC AUC (mean ± std) | F1 Score (mean ± std) |
|---|---|:---:|:---:|:---:|
| 1 | 0.1622 | 50.00 ± 0.00 | 0.4995 ± 0.0010 | 0.5333 ± 0.2667 |
| 2 | 0.4723 | 50.00 ± 0.00 | 0.5110 ± 0.0221 | 0.2667 ± 0.3266 |
| 3 | 0.7642 | 67.56 ± 3.29 | 0.7185 ± 0.0426 | 0.7236 ± 0.0181 |
| 4 | 1.0468 | 73.86 ± 1.72 | 0.7909 ± 0.0181 | 0.7775 ± 0.0135 |
| 5 | 1.3252 | 68.97 ± 10.69 | 0.7342 ± 0

--- Run 5 Final Metrics (N=12): Accuracy=82.28%, F1=0.8444 ---


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
import os
from glob import glob
from tqdm import tqdm
import random
from torch.utils.data import DataLoader 
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from collections import defaultdict 
import sys # Added for more robust printing during sweep

# --- GLOBAL CONSTANTS & HELPERS ---

# Function to set a fixed seed for all random operations for a specific run
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
# Global set_seed(42) removed for independent runs.

# Constants defining the filter bank span (Used in the sweep loop)
F_MIN = 500.0
F_MAX = 20000.0
# N_FFB, Q_CALCULATED, and center_freqs will be dynamically calculated 
# in the main sweep loop and must be accessible globally/via classes.

# Placeholder globals (will be overwritten in the main loop)
N_FFB = 1 
Q_CALCULATED = 1.0 
center_freqs = np.array([2000.0])

# ------------------------------
# FIXED FILTER BANK (FFB) Neuron
# ------------------------------
class FixedFilterBankNeuron(nn.Module):
    # This class will dynamically pull the global Q_CALCULATED value 
    # which is set correctly before ResonantNet instantiation in the main loop.
    
    def __init__(self, initial_f0, fixed_Q, duration=0.2, sample_rate=40000):
        super().__init__()
        self.register_buffer("omega0", torch.tensor(2.0 * np.pi * initial_f0, dtype=torch.float32))
        self.register_buffer("Q", torch.tensor(fixed_Q, dtype=torch.float32))
        self.register_buffer("t", torch.arange(0, duration, 1/sample_rate, dtype=torch.float32))

    def forward(self, freqs, amps, phases):
        # Ensure correct shape expansion for batch processing
        t_expanded = self.t.unsqueeze(0).unsqueeze(0).expand(freqs.size(0), -1, -1) 
        w_input = (2 * np.pi * freqs).unsqueeze(-1) 
        amps_expanded = amps.unsqueeze(-1) 
        phases_expanded = phases.unsqueeze(-1) 
        
        w0 = self.omega0
        Q = self.Q
        
        # Calculate frequency ratio once per input frequency set
        w_ratio = w_input.squeeze(-1) / w0 
        
        # Bandpass filter magnitude response calculation
        H_mag_numerator = (w_ratio / Q) 
        denominator_term = (w_ratio / Q)**2
        # Adding 1e-12 for numerical stability
        denominator = torch.sqrt((1 - w_ratio**2)**2 + denominator_term + 1e-12)
        H_mag = H_mag_numerator / denominator 
        H_mag_expanded = H_mag.unsqueeze(-1)
        
        # Calculate signal component for this neuron
        signal_components = (
            amps_expanded * H_mag_expanded * torch.sin(w_input * t_expanded + phases_expanded)
        )
        
        # Sum across input frequencies (dim=1) to get the time-domain signal
        output_signal = signal_components.sum(dim=1) 
        
        # Envelope extraction (absolute mean value)
        dc_output = torch.abs(output_signal).mean(dim=1)
        return dc_output

# ------------------------------
# ResonantNet (Modified for FFB)
# ------------------------------
class ResonantNet(nn.Module):
    # num_ffbs is the number of filters in the fixed bank (N)
    def __init__(self, num_ffbs, fixed_Q, hidden_size=16, clip_value=2.0): 
        super().__init__()
        # 1. Calculate f0s based on the input num_ffbs
        intermediate_points = num_ffbs + 2
        all_base_freqs = np.logspace(np.log10(F_MIN), np.log10(F_MAX), intermediate_points)
        base_freqs = all_base_freqs[1:-1]
        
        # 2. Instantiate the filter bank with the fixed Q
        self.ffbs = nn.ModuleList([
            FixedFilterBankNeuron(initial_f0=float(f0), fixed_Q=fixed_Q) for f0 in base_freqs
        ])
        
        CLIP_ACTIVATION = nn.Hardtanh(min_val=0.0, max_val=clip_value)
        
        # 3. MLP input size is determined by 'num_ffbs'
        self.net = nn.Sequential(
            nn.Linear(num_ffbs, hidden_size), CLIP_ACTIVATION,
            nn.Linear(hidden_size, hidden_size), CLIP_ACTIVATION,
            nn.Linear(hidden_size, 1) # Output is now logits
        )
        
    def forward(self, freqs_batch, amps_batch, phases_batch):
        # ffbs_features: list of [B] tensors
        ffbs_features = [ffb(freqs_batch, amps_batch, phases_batch) for ffb in self.ffbs]
        # Stack and transpose to get [B, N_FFB]
        x = torch.stack(ffbs_features).transpose(0, 1) 
        # Output is raw logits [B]
        return self.net(x).squeeze(1)

# ------------------------------
# Dataset (Unchanged)
# ------------------------------
class SignalDataset(torch.utils.data.Dataset):
    def __init__(self, root_folder="dataset_params", files_subset=None):
        self.files, self.labels = [], []
        pos_files = sorted(glob(os.path.join(root_folder, "positive", "*.json")))
        neg_files = sorted(glob(os.path.join(root_folder, "negative", "*.json")))
        
        if files_subset is not None:
            pos_files = [f for f in pos_files if f in files_subset]
            neg_files = [f for f in neg_files if f in files_subset]
            
        self.files += pos_files
        self.labels += [1] * len(pos_files)
        self.files += neg_files
        self.labels += [0] * len(neg_files)
        
        combined = sorted(zip(self.files, self.labels), key=lambda x: x[0])
        self.files = [f for f, l in combined]
        self.labels = [l for f, l in combined]

    def __len__(self):
        return len(self.files)
        
    def __getitem__(self, idx):
        with open(self.files[idx], 'r') as f: data = json.load(f)
        freqs = torch.tensor([d['freq'] for d in data], dtype=torch.float32)
        amps = torch.tensor([d['amp'] for d in data], dtype=torch.float32)
        phases = torch.tensor([d['phase'] for d in data], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return freqs, amps, phases, label

# ------------------------------
# Core Experiment Function (Encapsulated for 5-Run Average)
# ------------------------------
def run_experiment(run_id, num_epochs, batch_size, current_N, current_Q):
    """
    Performs one independent training and evaluation run for a specific N and Q.
    """
    # Set seed for this specific run
    set_seed(run_id) 
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    root_folder = "dataset_params"
    
    # 1. Get files and shuffle them separately
    all_pos_files = sorted(glob(os.path.join(root_folder, "positive", "*.json")))
    all_neg_files = sorted(glob(os.path.join(root_folder, "negative", "*.json")))

    random.shuffle(all_pos_files) 
    random.shuffle(all_neg_files) 

    # 2. Determine the size of the splits
    num_pos = len(all_pos_files)
    num_neg = len(all_neg_files)

    train_pos_split = num_pos // 2
    train_neg_split = num_neg // 2

    # 3. Create BALANCED splits
    train_files = all_pos_files[:train_pos_split] + all_neg_files[:train_neg_split]
    random.shuffle(train_files)
    eval_files = all_pos_files[train_pos_split:] + all_neg_files[train_neg_split:]
    random.shuffle(eval_files)

    # 4. Create Datasets and DataLoaders
    dataset_train = SignalDataset(root_folder=root_folder, files_subset=train_files)
    dataset_eval = SignalDataset(root_folder=root_folder, files_subset=eval_files)

    dataloader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
    dataloader_eval = DataLoader(dataset_eval, batch_size=batch_size, shuffle=False)
    
    # 5. Model, Optimizer, Loss - Pass current N and Q
    model = ResonantNet(num_ffbs=current_N, fixed_Q=current_Q, clip_value=2.0).to(device) 
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2) 
    criterion = nn.BCEWithLogitsLoss() 

    print(f"--- Starting Run {run_id+1} (N={current_N}, Q={current_Q:.4f})---", file=sys.stderr)

    # 6. Training Loop
    for epoch in range(num_epochs):
        model.train()
        
        for freqs, amps, phases, label in tqdm(dataloader_train, desc=f"Run {run_id+1}/N{current_N} - Epoch {epoch+1}/{num_epochs} (Train)"):
            freqs, amps, phases, label = freqs.to(device), amps.to(device), phases.to(device), label.to(device)
            
            optimizer.zero_grad()
            output = model(freqs, amps, phases)
            loss = criterion(output, label) 
            
            loss.backward()
            optimizer.step()

    # 7. Final Evaluation
    model.eval()
    all_labels = []
    all_probs = []
    all_preds = []
    eval_loss = 0.0

    with torch.no_grad():
        for freqs, amps, phases, label in dataloader_eval: 
            freqs, amps, phases, label = freqs.to(device), amps.to(device), phases.to(device), label.to(device)
            output = model(freqs, amps, phases)
            
            loss = criterion(output, label) 
            eval_loss += loss.item() * freqs.size(0)
            
            probabilities = torch.sigmoid(output)
            predicted = (probabilities >= 0.5).float()
            
            all_labels.append(label.cpu())
            all_probs.append(probabilities.cpu())
            all_preds.append(predicted.cpu())
            
    final_labels = torch.cat(all_labels).numpy()
    final_probs = torch.cat(all_probs).numpy()
    final_preds = torch.cat(all_preds).numpy()

    total_eval = len(final_labels)
    correct = (final_preds == final_labels).sum()

    # Handle potential metric errors
    try:
        final_auc = roc_auc_score(final_labels, final_probs)
        final_precision = precision_score(final_labels, final_preds, zero_division=0)
        final_recall = recall_score(final_labels, final_preds, zero_division=0)
        final_f1 = f1_score(final_labels, final_preds, zero_division=0)
    except ValueError:
        final_auc, final_precision, final_recall, final_f1 = np.nan, np.nan, np.nan, np.nan


    metrics = {
        'accuracy': correct / total_eval * 100,
        'loss': eval_loss / total_eval,
        'auc': final_auc,
        'precision': final_precision,
        'recall': final_recall,
        'f1': final_f1
    }

    print(f"--- Run {run_id+1} Final Metrics (N={current_N}): Accuracy={metrics['accuracy']:.2f}%, F1={metrics['f1']:.4f} ---", file=sys.stderr)
    return metrics


# ------------------------------
# Main Execution Block for SWEEP AVERAGING
# ------------------------------
BATCH_SIZE = 128
NUM_RUNS = 5
N_SWEEP_RANGE = range(1, 13) # N=1 to N=12 inclusive
ALL_SWEEP_RESULTS = [] # To store the final table data

print("="*80)
print(f"| **STARTING FFB ARCHITECTURE SWEEP: N={min(N_SWEEP_RANGE)} to N={max(N_SWEEP_RANGE)}**")
print("="*80)

for N in N_SWEEP_RANGE:
    # 1. Calculate N-dependent constants (Q and f0s)
    
    # Calculate Q according to the formula (Equation 42):
    # Q = 1 / ((F_MAX/F_MIN)^(1/(2*N)) - (F_MAX/F_MIN)^(-1/(2*N)))
    RATIO = F_MAX / F_MIN
    EXPONENT = 1.0 / (2.0 * N)
    # Handle the N=0 case defensively, though the loop starts at N=1
    if N == 0: 
        Q_CALCULATED = 1.0 
    else:
        Q_CALCULATED = 1.0 / (RATIO**EXPONENT - RATIO**(-EXPONENT))
    
    intermediate_points = N + 2
    all_base_freqs = np.logspace(np.log10(F_MIN), np.log10(F_MAX), intermediate_points)
    center_freqs = all_base_freqs[1:-1]
    
    # 2. Print Initial Parameters for the current N
    print("\n" + "#"*70)
    print(f"# **Fixed Filter Bank (FFB) Parameters for N={N}**")
    print("#" * 70)
    print(f"# Calculated Constant Quality Factor (Q): {Q_CALCULATED:.4f}")
    print(f"# Number of Filters (N): {N}")
    print("#" * 70)
    print("# Center Frequencies (f0, in Hz):")
    print(f"# " + ", ".join([f"{f:.2f}" for f in center_freqs]))
    print("#"*70)
    
    # 3. Execute the 5-run average logic
    all_metrics = defaultdict(list)

    for i in range(NUM_RUNS):
        # 'i' serves as the unique seed for each independent run
        # NOTE: num_epochs is set to 15 as per the user's provided code structure.
        metrics = run_experiment(i, num_epochs=15, batch_size=BATCH_SIZE, current_N=N, current_Q=Q_CALCULATED)
        
        for key, value in metrics.items():
            all_metrics[key].append(value)

    # 4. Calculate and display final statistics for this N
    print("\n" + "="*70)
    print(f"| **AVERAGE EVALUATION METRICS OVER {NUM_RUNS} RUNS (FFB - N={N})**")
    print("-" * 70)

    current_n_results = {'N': N} 

    for key, values in all_metrics.items():
        mean = np.mean(values)
        std = np.std(values)
        
        current_n_results[f'{key}_mean'] = mean
        current_n_results[f'{key}_std'] = std
        
        # Custom print formatting for metrics
        if key == 'accuracy':
            print(f"| Average Accuracy: {mean:.2f}% ± {std:.2f}%")
        elif key == 'loss':
            print(f"| Average Loss: {mean:.4f} ± {std:.4f}")
        else:
            print(f"| Average {key.upper()}: {mean:.4f} ± {std:.4f}")

    print("="*70)
    ALL_SWEEP_RESULTS.append(current_n_results)


# ------------------------------
# FINAL SWEEP RESULTS TABLE
# ------------------------------
print("\n\n" + "#" * 90)
print("# **COMPREHENSIVE SWEEP SUMMARY: FFB Performance (N=1 to N=12)**")
print("#" * 90)

# Print a final summary table in Markdown format
header = "| N | Q | Accuracy (mean ± std) | ROC AUC (mean ± std) | F1 Score (mean ± std) |"
separator = "|---|---|:---:|:---:|:---:|"
print(header)
print(separator)

for result in ALL_SWEEP_RESULTS:
    n = result['N']
    
    # Recalculate Q for the table (can't rely on stored result since it wasn't stored)
    if n == 0:
        q_value = 1.0
    else:
        EXPONENT = 1.0 / (2.0 * n)
        q_value = 1.0 / ((F_MAX/F_MIN)**EXPONENT - (F_MAX/F_MIN)**(-EXPONENT))
        
    acc = f"{result['accuracy_mean']:.2f} ± {result['accuracy_std']:.2f}"
    auc = f"{result['auc_mean']:.4f} ± {result['auc_std']:.4f}"
    f1 = f"{result['f1_mean']:.4f} ± {result['f1_std']:.4f}"
    
    print(f"| {n} | {q_value:.4f} | {acc} | {auc} | {f1} |")
    
print("#" * 90)

| **STARTING FFB ARCHITECTURE SWEEP: N=1 to N=12**

######################################################################
# **Fixed Filter Bank (FFB) Parameters for N=1**
######################################################################
# Calculated Constant Quality Factor (Q): 0.1622
# Number of Filters (N): 1
######################################################################
# Center Frequencies (f0, in Hz):
# 3162.28
######################################################################


--- Starting Run 1 (N=1, Q=0.1622)---
Run 1/N1 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:04<00:00, 33.95it/s]
--- Run 1 Final Metrics (N=1): Accuracy=50.00%, F1=0.6667 ---
--- Starting Run 2 (N=1, Q=0.1622)---
Run 2/N1 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:04<00:00, 33.93it/s]
--- Run 2 Final Metrics (N=1): Accuracy=50.00%, F1=0.0000 ---
--- Starting Run 3 (N=1, Q=0.1622)---
Run 3/N1 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:04<00:00, 33.75it/s]
--- Run 3 Final Metrics (N=1): Accuracy=50.00%, F1=0.6667 ---
--- Starting Run 4 (N=1, Q=0.1622)---
Run 4/N1 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:04<00:00, 33.56it/s]
--- Run 4 Final Metrics (N=1): Accuracy=50.00%, F1=0.6667 ---
--- Starting Run 5 (N=1, Q=0.1622)---
Run 5/N1 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:04<00:00, 33.64it/s]
--- Run 5 Final Metrics (N=1): Accuracy=50.00%, F1=0.6667 ---



| **AVERAGE EVALUATION METRICS OVER 5 RUNS (FFB - N=1)**
----------------------------------------------------------------------
| Average Accuracy: 50.00% ± 0.00%
| Average Loss: 0.6932 ± 0.0001
| Average AUC: 0.5000 ± 0.0001
| Average PRECISION: 0.4000 ± 0.2000
| Average RECALL: 0.8000 ± 0.4000
| Average F1: 0.5333 ± 0.2667

######################################################################
# **Fixed Filter Bank (FFB) Parameters for N=2**
######################################################################
# Calculated Constant Quality Factor (Q): 0.4723
# Number of Filters (N): 2
######################################################################
# Center Frequencies (f0, in Hz):
# 1709.98, 5848.04
######################################################################


--- Starting Run 1 (N=2, Q=0.4723)---
Run 1/N2 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:04<00:00, 32.69it/s]
--- Run 1 Final Metrics (N=2): Accuracy=50.00%, F1=0.0000 ---
--- Starting Run 2 (N=2, Q=0.4723)---
Run 2/N2 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:04<00:00, 33.06it/s]
--- Run 2 Final Metrics (N=2): Accuracy=50.00%, F1=0.6667 ---
--- Starting Run 3 (N=2, Q=0.4723)---
Run 3/N2 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:04<00:00, 33.36it/s]
--- Run 3 Final Metrics (N=2): Accuracy=50.00%, F1=0.0000 ---
--- Starting Run 4 (N=2, Q=0.4723)---
Run 4/N2 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:04<00:00, 33.15it/s]
--- Run 4 Final Metrics (N=2): Accuracy=50.00%, F1=0.0000 ---
--- Starting Run 5 (N=2, Q=0.4723)---
Run 5/N2 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:04<00:00, 33.22it/s]
--- Run 5 Final Metrics (N=2): Accuracy=50.00%, F1=0.6667 ---



| **AVERAGE EVALUATION METRICS OVER 5 RUNS (FFB - N=2)**
----------------------------------------------------------------------
| Average Accuracy: 50.00% ± 0.00%
| Average Loss: 0.6932 ± 0.0000
| Average AUC: 0.5033 ± 0.0071
| Average PRECISION: 0.2000 ± 0.2449
| Average RECALL: 0.4000 ± 0.4899
| Average F1: 0.2667 ± 0.3266

######################################################################
# **Fixed Filter Bank (FFB) Parameters for N=3**
######################################################################
# Calculated Constant Quality Factor (Q): 0.7642
# Number of Filters (N): 3
######################################################################
# Center Frequencies (f0, in Hz):
# 1257.43, 3162.28, 7952.71
######################################################################


...

--- Starting Run 1 (N=11, Q=2.9680)---
Run 1/N11 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:07<00:00, 20.19it/s]
--- Run 1 Final Metrics (N=11): Accuracy=78.17%, F1=0.8055 ---
--- Starting Run 2 (N=11, Q=2.9680)---
Run 2/N11 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:07<00:00, 19.63it/s]
--- Run 2 Final Metrics (N=11): Accuracy=78.52%, F1=0.8178 ---
--- Starting Run 3 (N=11, Q=2.9680)---
Run 3/N11 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:07<00:00, 20.25it/s]
--- Run 3 Final Metrics (N=11): Accuracy=78.18%, F1=0.8151 ---
--- Starting Run 4 (N=11, Q=2.9680)---
Run 4/N11 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:07<00:00, 19.65it/s]
--- Run 4 Final Metrics (N=11): Accuracy=77.73%, F1=0.8025 ---
--- Starting Run 5 (N=11, Q=2.9680)---
Run 5/N11 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:07<00:00, 19.73it/s]
--- Run 5 Final Metrics (N=11): Accuracy=79.62%, F1=0.8259 ---



| **AVERAGE EVALUATION METRICS OVER 5 RUNS (FFB - N=11)**
----------------------------------------------------------------------
| Average Accuracy: 78.45% ± 0.64%
| Average Loss: 0.4503 ± 0.0080
| Average AUC: 0.8396 ± 0.0090
| Average PRECISION: 0.7171 ± 0.0072
| Average RECALL: 0.9403 ± 0.0293
| Average F1: 0.8134 ± 0.0085

######################################################################
# **Fixed Filter Bank (FFB) Parameters for N=12**
######################################################################
# Calculated Constant Quality Factor (Q): 3.2402
# Number of Filters (N): 12
######################################################################
# Center Frequencies (f0, in Hz):
# 664.06, 881.94, 1171.32, 1555.65, 2066.08, 2743.99, 3644.33, 4840.09, 6428.19, 8537.36, 11338.59, 15058.95
######################################################################


--- Starting Run 1 (N=12, Q=3.2402)---
Run 1/N12 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:08<00:00, 17.65it/s]
--- Run 1 Final Metrics (N=12): Accuracy=74.30%, F1=0.7423 ---
--- Starting Run 2 (N=12, Q=3.2402)---
Run 2/N12 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:08<00:00, 18.34it/s]
--- Run 2 Final Metrics (N=12): Accuracy=69.27%, F1=0.6377 ---
--- Starting Run 3 (N=12, Q=3.2402)---
Run 3/N12 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:08<00:00, 18.16it/s]
--- Run 3 Final Metrics (N=12): Accuracy=76.34%, F1=0.7962 ---
--- Starting Run 4 (N=12, Q=3.2402)---
Run 4/N12 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:08<00:00, 18.17it/s]
--- Run 4 Final Metrics (N=12): Accuracy=78.50%, F1=0.8047 ---
--- Starting Run 5 (N=12, Q=3.2402)---
Run 5/N12 - Epoch 15/15 (Train): 100%|██████████| 157/157 [00:08<00:00, 18.18it/s]



| **AVERAGE EVALUATION METRICS OVER 5 RUNS (FFB - N=12)**
----------------------------------------------------------------------
| Average Accuracy: 74.95% ± 3.13%
| Average Loss: 0.4999 ± 0.0486
| Average AUC: 0.8276 ± 0.0071
| Average PRECISION: 0.7304 ± 0.0305
| Average RECALL: 0.8065 ± 0.1505
| Average F1: 0.7560 ± 0.0633


##########################################################################################
# **COMPREHENSIVE SWEEP SUMMARY: FFB Performance (N=1 to N=12)**
##########################################################################################
| N | Q | Accuracy (mean ± std) | ROC AUC (mean ± std) | F1 Score (mean ± std) |
|---|---|:---:|:---:|:---:|
| 1 | 0.1622 | 50.00 ± 0.00 | 0.5000 ± 0.0001 | 0.5333 ± 0.2667 |
| 2 | 0.4723 | 50.00 ± 0.00 | 0.5033 ± 0.0071 | 0.2667 ± 0.3266 |
| 3 | 0.7642 | 50.00 ± 0.00 | 0.4999 ± 0.0001 | 0.2667 ± 0.3266 |
| 4 | 1.0468 | 53.02 ± 3.71 | 0.5346 ± 0.0425 | 0.3984 ± 0.3255 |
| 5 | 1.3252 | 50.00 ± 0.00 | 0.5000 ± 0.0000 | 0

--- Run 5 Final Metrics (N=12): Accuracy=76.36%, F1=0.7992 ---
